# ANALISIS DEL DATASET
En este notebook solo se analizará el dataset original sin variables exogenas agregadas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf

## Pre-procesamiento del DATASET


In [ ]:
# -----------------------------
# 1) Carga y limpieza inicial
# -----------------------------
df = pd.read_csv("../data/raw/demandas_horarias_2023_2026.csv", encoding="utf-8")

# Normaliza nombres de columnas para evitar problemas por espacios o saltos de línea
df.columns = (
    df.columns.astype(str)
      .str.replace("\n", " ", regex=False)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

# -----------------------------
# 2) Selección mínima de columnas
# -----------------------------
# Nos quedamos solo con FECHA, Horario y Total
df = df[["FECHA", "Horario", "TOTAL_SIP"]].copy()

# Convierte FECHA a datetime
df["FECHA"] = pd.to_datetime(df["FECHA"], errors="coerce")

# Limpia Horario y lo convierte a formato hora
df["Horario"] = df["Horario"].astype(str).str.strip()

# -----------------------------
# 3) Filtrado: solo horas exactas y Total válido
# -----------------------------
# Horas exactas: HH:00:00 o HH:00
df = df[df["Horario"].str.match(r"^\d{1,2}:00(:00)?$")].copy()

# Total numérico
df["TOTAL_SIP"] = pd.to_numeric(df["TOTAL_SIP"], errors="coerce")

# Elimina nulos y ceros
df = df[df["TOTAL_SIP"].notna() & (df["TOTAL_SIP"] > 0)].copy()

# -----------------------------
# 4) Construcción de datetime
# -----------------------------
# Une fecha + hora
df["datetime"] = pd.to_datetime(
    df["FECHA"].dt.strftime("%Y-%m-%d") + " " + df["Horario"].str.replace(" ", ""),
    errors="coerce"
)

# Elimina filas inválidas
df = df[df["datetime"].notna()].copy()

# Ordena y quita duplicados por timestamp
df = df.sort_values("datetime").drop_duplicates(subset="datetime")

# Deja la serie lista
df = df.set_index("datetime")[["TOTAL_SIP"]].rename(columns={"TOTAL_SIP": "demanda"})

# -----------------------------
# 5) Serie horaria continua
# -----------------------------
# Reindexa a frecuencia horaria para detectar huecos
full_index = pd.date_range(df.index.min(), df.index.max(), freq="h")
df = df.reindex(full_index)

# Interpola huecos pequeños si aparecen tras el reindex
df["demanda"] = df["demanda"].interpolate(method="time")

# Si quedaron nulos extremos, se eliminan
df = df.dropna(subset=["demanda"])

df = df.loc[: "2026-01-01 00:00:00"]

```markdown
### Análisis de la Tendencia Estructural de la Demanda
```


In [ ]:
from statsmodels.tsa.seasonal import MSTL
from sklearn.linear_model import TheilSenRegressor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) Descomposición de tendencia usando el método robusto STL (Seasonal-Trend decomposition using Loess), con un periodo estacional definido como una semana completa en horas (24*7). Se utiliza la opción 'robust=True' para reducir la influencia de valores outliers.
mstl = MSTL(
    df["demanda"],
    periods=(24, 24*7, 24*365)
)
resultado = mstl.fit()

demanda_suavizada = (
    df["demanda"]
    .rolling(
        window=24*7,
        center=True
    )
    .mean()
)
# Extraemos solo la componente de tendencia estimada por el modelo y eliminamos los NaN que aparecen al inicio o final debido a la descomposición estacional.
trend = resultado.trend.dropna()

# 2) Ajuste de un modelo de regresión robusto (Theil-Sen, no paramétrico y resistente a outliers) sobre toda la serie temporal extraída para estimar una recta lineal que capture el crecimiento o decrecimiento estructural.
X = np.arange(len(trend)).reshape(-1, 1)
y = trend.values

modelo = TheilSenRegressor(random_state=0)
modelo.fit(X, y)

# Obtenemos la predicción de la tendencia lineal sobre los mismos puntos temporales para visualizarla junto a la tendencia STL.
recta = modelo.predict(X)
pendiente_hora = modelo.coef_[0] # el primer coeficiente es la pendiente en MW/hora
pendiente_anual = pendiente_hora * 24 * 365.25

print(f"Pendiente: {pendiente_hora:.6f} MW/hora")
print(f"Pendiente anual equivalente: {pendiente_anual:.2f} MW/año")

# 3) Visualización de la tendencia descompuesta por STL y su aproximación lineal robusta para comparar el comportamiento estructural a largo plazo.
plt.figure(figsize=(15, 5))
plt.plot(trend.index, demanda_suavizada, label="Demanda")
plt.plot(trend.index, recta, linewidth=3, label="Tendencia")
plt.legend()
plt.title("Tendencia estructural de la demanda")
plt.ylabel("MW")
plt.show()

# Resumen anual agregado por año calendario para obtener métricas clave como media, mediana y máximo. Esto ayuda a entender el comportamiento estacionalidad-correcto del sistema eléctrico.
resumen_anual = (
    df["demanda"]
    .resample("YE")  # Agrupa los datos anualmente en base al índice temporal de la serie original 'df'
    .agg(
        media="mean",
        mediana="median",
        maximo="max"
    )
)
print(resumen_anual)


In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# ESTACIONALIDAD DIARIA (VERANO VS INVIERNO)
# ============================================================

verano = df[df.index.month.isin([12, 1, 2])]
invierno = df[df.index.month.isin([6, 7, 8])]

perfil_verano = (
    verano
    .groupby(verano.index.hour)["demanda"]
    .mean()
)

perfil_invierno = (
    invierno
    .groupby(invierno.index.hour)["demanda"]
    .mean()
)

amp_diaria_verano = (
    perfil_verano.max()
    - perfil_verano.min()
)

amp_diaria_invierno = (
    perfil_invierno.max()
    - perfil_invierno.min()
)

print(f"Amplitud diaria verano: {amp_diaria_verano:.2f} MW")
print(f"Amplitud diaria invierno: {amp_diaria_invierno:.2f} MW")

plt.figure(figsize=(12,5))
plt.plot(perfil_invierno.index, perfil_invierno, label="Invierno")
plt.plot(perfil_verano.index, perfil_verano, label="Verano")
plt.xlabel("Hora del día")
plt.ylabel("Demanda (MW)")
plt.title("Perfil diario promedio")
plt.legend()
plt.show()


# ============================================================
# ESTACIONALIDAD SEMANAL
# ============================================================

perfil_semanal = (
    df
    .groupby(df.index.dayofweek)["demanda"]
    .mean()
)

perfil_semanal.index = [
    "Lun", "Mar", "Mié",
    "Jue", "Vie", "Sáb", "Dom"
]

amp_semanal = (
    perfil_semanal.max()
    - perfil_semanal.min()
)

print(f"Amplitud semanal: {amp_semanal:.2f} MW")

plt.figure(figsize=(10,5))
plt.plot(
    perfil_semanal.index,
    perfil_semanal.values,
    marker="o"
)
plt.ylabel("Demanda (MW)")
plt.title("Perfil semanal promedio")
plt.show()

# ============================================================
# ESTACIONALIDAD ANUAL
# ============================================================

perfil_anual = (
    df
    .groupby(df.index.month)["demanda"]
    .mean()
)

amp_anual = (
    perfil_anual.max()
    - perfil_anual.min()
)

print(f"Amplitud anual: {amp_anual:.2f} MW")

plt.figure(figsize=(12,5))
plt.plot(
    perfil_anual.index,
    perfil_anual.values,
    marker="o"
)

plt.xticks(range(1,13))
plt.xlabel("Mes")
plt.ylabel("Demanda (MW)")
plt.title("Perfil anual promedio")
plt.show()


# ============================================================
# RESUMEN
# ============================================================

print("\nResumen de estacionalidad")
print(f"Diaria (verano):  {amp_diaria_verano:.2f} MW")
print(f"Diaria (invierno): {amp_diaria_invierno:.2f} MW")
print(f"Semanal:          {amp_semanal:.2f} MW")
print(f"Anual:            {amp_anual:.2f} MW")

In [ ]:
# ============================================================
# CURVAS DE PROMEDIO POR AÑO + PROMEDIO GENERAL
# ============================================================

# Etiquetas de los días de la semana
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

# Excluye 2026 (año incompleto)
df_plot = df[df.index.year != 2026]

# Años presentes en la serie
años = sorted(df_plot.index.year.unique())

# ------------------------------------------------------------
# 1) PERFIL DIARIO (hora del día) por año + promedio general
#    diferenciando VERANO vs INVIERNO
# ------------------------------------------------------------
meses_verano = [12, 1, 2]
meses_invierno = [6, 7, 8]

df_verano = df_plot[df_plot.index.month.isin(meses_verano)]
df_invierno = df_plot[df_plot.index.month.isin(meses_invierno)]

perfil_diario_verano_anual = (
    df_verano.groupby([df_verano.index.year, df_verano.index.hour])["demanda"]
      .mean()
      .unstack(0)
)
perfil_diario_invierno_anual = (
    df_invierno.groupby([df_invierno.index.year, df_invierno.index.hour])["demanda"]
      .mean()
      .unstack(0)
)

perfil_diario_verano_general = df_verano.groupby(df_verano.index.hour)["demanda"].mean()
perfil_diario_invierno_general = df_invierno.groupby(df_invierno.index.hour)["demanda"].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Subplot VERANO
ax = axes[0]
for año in años:
    ax.plot(perfil_diario_verano_anual.index, perfil_diario_verano_anual[año], label=f"{año}")
ax.plot(
    perfil_diario_verano_general.index,
    perfil_diario_verano_general,
    color="black",
    linewidth=3,
    label="Promedio general"
)
ax.set_xlabel("Hora del día")
ax.set_ylabel("Demanda (MW)")
ax.set_title("Perfil diario promedio por año - VERANO")
ax.legend()

# Subplot INVIERNO
ax = axes[1]
for año in años:
    ax.plot(perfil_diario_invierno_anual.index, perfil_diario_invierno_anual[año], label=f"{año}")
ax.plot(
    perfil_diario_invierno_general.index,
    perfil_diario_invierno_general,
    color="black",
    linewidth=3,
    label="Promedio general"
)
ax.set_xlabel("Hora del día")
ax.set_title("Perfil diario promedio por año - INVIERNO")
ax.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 2) PERFIL SEMANAL (día de la semana) por año + promedio general
# ------------------------------------------------------------
perfil_semanal_anual = (
    df_plot.groupby([df_plot.index.year, df_plot.index.dayofweek])["demanda"]
      .mean()
      .unstack(0)
)
perfil_semanal_general = df_plot.groupby(df_plot.index.dayofweek)["demanda"].mean()

plt.figure(figsize=(10, 5))
for año in años:
    plt.plot(dias, perfil_semanal_anual[año], marker="o", label=f"{año}")
plt.plot(
    dias,
    perfil_semanal_general,
    color="black",
    linewidth=3,
    marker="o",
    label="Promedio general"
)
plt.xlabel("Día de la semana")
plt.ylabel("Demanda (MW)")
plt.title("Perfil semanal promedio por año")
plt.legend()
plt.show()

# ------------------------------------------------------------
# 3) PERFIL ANUAL (mes) por año + promedio general
# ------------------------------------------------------------
perfil_anual_por_año = (
    df_plot.groupby([df_plot.index.year, df_plot.index.month])["demanda"]
      .mean()
      .unstack(0)
)
perfil_anual_general = df_plot.groupby(df_plot.index.month)["demanda"].mean()

plt.figure(figsize=(12, 5))
for año in años:
    plt.plot(perfil_anual_por_año.index, perfil_anual_por_año[año], marker="o", label=f"{año}")
plt.plot(
    perfil_anual_general.index,
    perfil_anual_general,
    color="black",
    linewidth=3,
    marker="o",
    label="Promedio general"
)
plt.xticks(range(1, 13))
plt.xlabel("Mes")
plt.ylabel("Demanda (MW)")
plt.title("Perfil anual promedio por año")
plt.legend()
plt.show()

In [ ]:
# ============================================================
# ACF DE LA DEMANDA (lags 1 a 168)
# ============================================================

from statsmodels.tsa.stattools import acf
import numpy as np

# Calcula la ACF para lags 1 a 168 (descarta el lag 0)
lags = np.arange(1, 169)
acf_values = acf(df["demanda"], nlags=168)[1:]

# No mostrar valores negativos: se recortan a 0
acf_pos = np.clip(acf_values, 0, None)

# Pico máximo y su lag
lag_pico = int(lags[np.argmax(acf_values)])
valor_pico = acf_values.max()

# Valor en el lag 24 (índice 23 porque lags empieza en 1)
lag_24 = 24
valor_24 = acf_values[lag_24 - 1]

fig, ax = plt.subplots(figsize=(15, 5))
ax.vlines(lags, 0, acf_pos, linewidth=1.2)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Lag (horas)")
ax.set_ylabel("Autocorrelación")
ax.set_title("ACF de la demanda (lags 1 a 168)")

# Resalta y anota el pico
ax.plot(lag_pico, valor_pico, "ro", markersize=6)
ax.annotate(
    f"lag {lag_pico} = {valor_pico:.3f}",
    xy=(lag_pico, valor_pico),
    xytext=(lag_pico + 12, valor_pico - 0.08),
    arrowprops=dict(arrowstyle="->", color="red"),
    color="red",
    fontsize=10
)

# Resalta y anota el lag 24
ax.plot(lag_24, valor_24, "go", markersize=6)
ax.annotate(
    f"lag {lag_24} = {valor_24:.3f}",
    xy=(lag_24, valor_24),
    xytext=(lag_24 + 12, valor_24 - 0.08),
    arrowprops=dict(arrowstyle="->", color="green"),
    color="green",
    fontsize=10
)

lags_medios = [168]
media_y = np.mean([acf_values[l - 1] for l in lags_medios])
ax.axhline(media_y, color="orange", linestyle="-", linewidth=1.5)

plt.show()
print(f"Pico de autocorrelación: lag {lag_pico} (valor {valor_pico:.3f})")
print(f"Lag 24: valor {valor_24:.3f}")

In [ ]:
residual = resultado.resid.dropna() # residuo obtenido por STL

porcentaje_ruido = (
    np.var(residual) /
    np.var(df["demanda"])
) * 100

print(f"Residuo diario explicado: {porcentaje_ruido:.2f}%") 
# temperatura
# verano/invierno
# feriados
# eventos especiales
# crecimiento económico

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
adf = adfuller(residual)

# ADF:
# - H0: no estacionaria
# - H1: estacionaria

# KPSS:
# - H0: estacionaria
# - H1: no estacionaria

print("ADF:", adf[0])
print("p-value ADF:", adf[1])

stat, p, *_ = kpss(residual)

print("p-value KPSS:", p)

In [ ]:

plot_acf(residual, lags=200)
plot_pacf(residual, lags=200)
# Calcular PACF
pacf_values = pacf(residual, nlags=200)

# Mostrar los primeros valores
for lag, value in enumerate(pacf_values[:168]):
    if (value > 0.2):
        print(f"Lag {lag}: {value:.4f}")